# The Automated Enterprise Researcher

This notebook demonstrates the Capstone Architecture combining LangChain RAG with CrewAI Orchestration.

In [ ]:
import os
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import Chroma
from langchain_openai import OpenAIEmbeddings
from crewai import Agent, Task, Crew, Process

# Ensure your OPENAI_API_KEY is available in the environment
# os.environ["OPENAI_API_KEY"] = "sk-..."

print("1. Starting LangChain RAG Retrieval...")

# For demonstration, we simulate the 'PyPDFLoader' retrieving a block of text 
# (In a real scenario, you'd load a physical PDF here)
business_context = """
# Retrieved Q3 Enterprise Data
- Revenue grew by 14% to $4.2M.
- Churn increased from 1.2% to 3.4% in the European market due to competitor pricing drops.
- A critical cybersecurity vulnerability was detected but patched before any data breach occurred.
"""

print("Data retrieved via LangChain! Handing off to CrewAI...")

# ----------------------------------------
# 2. CREW AI Orchestration Layer
# ----------------------------------------

# Define the analytic agent
risk_analyst = Agent(
    role='Senior Risk Analyst',
    goal='Identify and synthesize business risks from raw enterprise data.',
    backstory='You are a meticulous auditor who spots devastating vulnerabilities hidden in quarterly reports.',
    verbose=True,
    allow_delegation=False,
    llm="gpt-4-turbo"
)

# Define the writing agent
exec_writer = Agent(
    role='Executive Briefing Writer',
    goal='Write a calm, authoritative executive summary based on the risk analysis.',
    backstory='You write direct, no-nonsense briefs for the CEO. You take complex panicked reports and make them actionable.',
    verbose=True,
    allow_delegation=False,
    llm="gpt-4-turbo"
)

# Pass the LangChain Context explicitly into the CrewAI Task
analysis_task = Task(
    description=f'Analyze the following retrieved data block and list top 3 action items regarding risk: {business_context}',
    expected_output='A bulleted list of 3 specific risks/action items based strictly on the text.',
    agent=risk_analyst
)

writing_task = Task(
    description='Using the analyst\'s output, write a 2-paragraph memo to the CEO. Do NOT hallucinate external data.',
    expected_output='A 2-paragraph markdown memo.',
    agent=exec_writer
)

# Execute the Crew
enterprise_crew = Crew(
    agents=[risk_analyst, exec_writer],
    tasks=[analysis_task, writing_task],
    process=Process.sequential 
)

print("\n2. Executing CrewAI Master Pipeline...\n")
final_report = enterprise_crew.kickoff()

print("\n######################\nFINAL CEO BRIEFING\n######################")
print(final_report)
